# Explore performance evaluation summary results

In [208]:
import json
from pathlib import Path
import pandas as pd

LOGS_DIR = Path("../logs")

# ------------------------------------------------------------------
# Helper: conditional IPSNR (mean over samples with PP IoU > 0, plus n)
# ------------------------------------------------------------------
def conditional_ipsnr(csv_path, ipsnr_col="pp_ipsnr", iou_col="pp_iou"):
    if not csv_path.exists():
        return "N/A"
    df = pd.read_csv(csv_path)
    if ipsnr_col not in df.columns or iou_col not in df.columns:
        return "N/A"
    
    total_samples = len(df)
    subset = df.loc[df[iou_col] > 0, ipsnr_col]
    
    if len(subset) == 0 or total_samples == 0:
        return "N/A"
    
    # Fraction format: Mean (valid/total)
    return f"{subset.mean():.2f} ({len(subset)}/{total_samples})"


GROUPS = [
    {
        "header": "Direct (GOES->Landsat) - HAUNet",
        "order": [
            "e1_haunet_direct_focal_dice",
            "e1_haunet_direct_focal_dice_no_cov_encoder",
            "e1_haunet_direct_focal_dice_no_dem",
            "e1_haunet_direct_focal_dice_no_landcover",
            "e1_haunet_direct_focal_dice_dilated",
        ],
        "labels": {
            "e1_haunet_direct_focal_dice":               "E1: Full, No Target Dilation",
            "e1_haunet_direct_focal_dice_no_cov_encoder": "E1a: No Cov. Pyramid Enc.",
            "e1_haunet_direct_focal_dice_no_dem":         "E1b: No DEM",
            "e1_haunet_direct_focal_dice_no_landcover":   "E1c: No Land Cover",
            "e1_haunet_direct_focal_dice_dilated":        "E1d: Dilated Target",
        },
    },
    {
        "header": "Cascaded (GOES->VIIRS->Landsat) - HAUNet",
        "order": [
            "e2_haunet_cascaded_stage1_focal_dice",
            "e2_haunet_cascaded_stage1_focal_dice_dilated",
            "e2_haunet_cascaded_stage2_focal_dice_single_dilated",
            "e2_haunet_cascaded_stage2_focal_dice_double_dilated",
        ],
        "labels": {
            "e2_haunet_cascaded_stage1_focal_dice":                "E2.1a: Stage 1 (no dilation)",
            "e2_haunet_cascaded_stage1_focal_dice_dilated":        "E2.1b: Stage 1 (dilated)",
            "e2_haunet_cascaded_stage2_focal_dice_single_dilated": "E2.2a: Stage 2 (E2.1a input, dilated)",
            "e2_haunet_cascaded_stage2_focal_dice_double_dilated": "E2.2b: Stage 2 (E2.1b input dilated)",
        },
        # Split "best" comparison: Stage 1 rows compared only against each
        # other, Stage 2 rows compared only against each other.
        "subgroups": [
            ["e2_haunet_cascaded_stage1_focal_dice", "e2_haunet_cascaded_stage1_focal_dice_dilated"],
            ["e2_haunet_cascaded_stage2_focal_dice_single_dilated", "e2_haunet_cascaded_stage2_focal_dice_double_dilated"],
        ],
    },
    {
        "header": "Direct (GOES->Landsat) - Autoencoder",
        "order": [
            "e3_autoencoder_direct_focal_dice",
            "e3_autoencoder_direct_focal_dice_dilated",
        ],
        "labels": {
            "e3_autoencoder_direct_focal_dice":         "E3: Full, No Target Dilation",
            "e3_autoencoder_direct_focal_dice_dilated": "E3d: Dilated Target",
        },
    },
    {
        "header": "Cascaded pipeline - Autoencoder",
        "order": [
            "e4_autoencoder_cascaded_stage1_focal_dice",
            "e4_autoencoder_cascaded_stage2_focal_dice",
        ],
        "labels": {
            "e4_autoencoder_cascaded_stage1_focal_dice": "E4.1: Stage 1 (no dilation)",
            "e4_autoencoder_cascaded_stage2_focal_dice": "E4.2: Stage 2 (dilated targets)",
        },
        "subgroups": [
            ["e4_autoencoder_cascaded_stage1_focal_dice"],
            ["e4_autoencoder_cascaded_stage2_focal_dice"],
        ],
    },
]

COLUMN_ORDER = ["auprc", "iou (Post-PP)", "precision (Post-PP)", "recall (Post-PP)", "boundary_f1 (Post-PP)", "ipsnr"]
COLUMN_LABELS = {
    "auprc": "AUPRC",
    "iou (Post-PP)": "IoU",
    "precision (Post-PP)": "Precision",
    "recall (Post-PP)": "Recall",
    "boundary_f1 (Post-PP)": "Boundary F1",
    "ipsnr": "IPSNR (n)",
}

In [209]:
# ------------------------------------------------------------------
# Load every summary JSON once, keyed by experiment name
# ------------------------------------------------------------------
all_json_files = sorted(LOGS_DIR.glob("*best_val_auprc*.json"))
experiments = {}
baseline_all_zero = {}
baseline_prevalence = {}

for file in all_json_files:
    with open(file, "r") as f:
        content = json.load(f)
    exp_name = content.get("experiment", "")
    metrics = content.get("metrics", {})
    per_sample_csv = LOGS_DIR / file.name.replace("_test_summary.json", "_test_per_sample.csv")

    experiments[exp_name] = {
        "auprc": metrics.get("auprc", {}).get("mean"),
        "iou (Post-PP)": metrics.get("pp_iou", {}).get("mean"),
        "precision (Post-PP)": metrics.get("pp_precision", {}).get("mean"),
        "recall (Post-PP)": metrics.get("pp_recall", {}).get("mean"),
        "boundary_f1 (Post-PP)": metrics.get("pp_boundary_f1", {}).get("mean"),
        "ipsnr": conditional_ipsnr(per_sample_csv),
    }

    if not baseline_all_zero:
        b_zero = content.get("baseline_all_zero", {})
        baseline_all_zero = {
            "auprc": b_zero.get("auprc", {}).get("mean"),
            "iou (Post-PP)": b_zero.get("iou", {}).get("mean"),
            "precision (Post-PP)": b_zero.get("precision", {}).get("mean"),
            "recall (Post-PP)": b_zero.get("recall", {}).get("mean"),
            "boundary_f1 (Post-PP)": None,
            "ipsnr": None,
        }
    if not baseline_prevalence:
        b_prev = content.get("baseline_prevalence_rate", {})
        baseline_prevalence = {
            "auprc": b_prev.get("auprc", {}).get("mean"),
            "iou (Post-PP)": b_prev.get("iou", {}).get("mean"),
            "precision (Post-PP)": b_prev.get("precision", {}).get("mean"),
            "recall (Post-PP)": b_prev.get("recall", {}).get("mean"),
            "boundary_f1 (Post-PP)": None,
            "ipsnr": None,
        }

In [210]:
# ------------------------------------------------------------------
# Build LaTeX body manually: section header rows + data rows
# ------------------------------------------------------------------
def fmt(val, is_best=False):
    if val is None or (isinstance(val, float) and pd.isna(val)):
        return "N/A"
    if isinstance(val, str):
        return f"\\textbf{{{val}}}" if is_best else val
    return f"\\textbf{{{val:.4f}}}" if is_best else f"{val:.4f}"

lines = []
lines.append(r"\begin{table}[htbp]")
lines.append(r"\centering")
lines.append(r"\footnotesize")
lines.append(r"\caption{Model performance evaluation across feature ablations vs. fixed baselines. Precision, Recall, Boundary F1, and IPSNR are computed on Otsu-thresholded predictions. IPSNR is reported as the mean over samples with nonzero post-processed IoU, with the contributing sample count in parentheses. Bold indicates the best value for that metric among the shown ablations of the same architecture and target}")
lines.append(r"\label{tab:all_results}")
lines.append(r"\begin{tabularx}{\textwidth}{lYYYYYY}")
lines.append(r"\hline")
header_row = " & ".join(["\\textbf{Exp.}"] + [f"\\textbf{{{COLUMN_LABELS[c]}}}" for c in COLUMN_ORDER])
lines.append(header_row + r" \\ \hline")

for i, group in enumerate(GROUPS):
    if i > 0:
        lines.append(r"\hline")
    lines.append(f"\\multicolumn{{{len(COLUMN_ORDER)+1}}}{{l}}{{\\textit{{{group['header']}}}}} \\\\")
    partitions = group.get("subgroups", [group["order"]])
    best_per_col = {}

    for sub in partitions:
        sub_rows = {eid: experiments[eid] for eid in sub if eid in experiments}
        # Only compute a "best" if there's more than one valid experiment
        # to compare — a single-row subgroup has no ablation comparison,
        # so nothing should be bolded.
        has_comparison = len(sub_rows) > 1
        for col in COLUMN_ORDER:
            if has_comparison:
                vals = [r.get(col) for r in sub_rows.values()
                         if isinstance(r.get(col), (int, float)) and not pd.isna(r.get(col))]
                max_val = max(vals) if vals else None
            else:
                max_val = None  # no comparison possible -> never bold
            for eid in sub:
                best_per_col.setdefault(eid, {})[col] = max_val

    for eid in group["order"]:
        if eid not in experiments:
            print(f"[warn] missing data for {eid}")
            continue
        row = experiments[eid]
        label = group["labels"].get(eid, eid.replace("_", r"\_"))
        cells = [label]
        for col in COLUMN_ORDER:
            v = row.get(col)
            # Safely fetch target best for this specific experiment ID
            target_best = best_per_col.get(eid, {}).get(col)
            is_best = (col != "ipsnr") and (target_best is not None) and (v == target_best)
            cells.append(fmt(v, is_best=is_best))
        lines.append(" & ".join(cells) + r" \\")

# Baseline rows
lines.append(r"\hline")    
for name, b in [("Baseline: All-Zero", baseline_all_zero), ("Baseline: Prevalence-Rate", baseline_prevalence)]:
    cells = [name] + [fmt(b.get(col)) for col in COLUMN_ORDER]
    lines.append(" & ".join(cells) + r" \\")

lines.append(r"\hline")
lines.append(r"\end{tabularx}")
lines.append(r"\end{table}")

full_latex_table = "\n".join(lines)

In [211]:
# ------------------------------------------------------------------
# Cross-group summary: best experiment per GROUP (by primary metric),
# for direct architecture/pipeline comparison. E2 (cascaded HAUNet)
# mixes Stage 1 (VIIRS target) and Stage 2 (Landsat target) experiments,
# which predict different things and aren't directly comparable — so it
# is split into two summary rows instead of one.
# ------------------------------------------------------------------
PRIMARY_METRIC = "auprc"

def best_in(eids):
    valid = {eid: experiments[eid] for eid in eids
             if eid in experiments
             and isinstance(experiments[eid].get(PRIMARY_METRIC), (int, float))
             and not pd.isna(experiments[eid].get(PRIMARY_METRIC))}
    if not valid:
        return None
    return max(valid, key=lambda k: valid[k][PRIMARY_METRIC])

summary_specs = [
    ("Direct - HAUNet",            [eid for eid in GROUPS[0]["order"]], GROUPS[0]["labels"]),
    ("Cascaded Stage 1 - HAUNet",  GROUPS[1]["subgroups"][0],           GROUPS[1]["labels"]),
    ("Cascaded Stage 2 - HAUNet",  GROUPS[1]["subgroups"][1],           GROUPS[1]["labels"]),
    ("Direct - Autoencoder",       GROUPS[2]["order"],                  GROUPS[2]["labels"]),
    ("Cascaded - Autoencoder",     GROUPS[3]["order"],                  GROUPS[3]["labels"]),
]

summary_rows = []
for header, eids, labels in summary_specs:
    best_eid = best_in(eids)
    if best_eid is None:
        print(f"[warn] no valid data for summary group: {header}")
        continue
    summary_rows.append((header, labels.get(best_eid, best_eid.replace("_", r"\_")), experiments[best_eid]))

# Bold the best value per column, across these summary rows only
summary_best_per_col = {}
for col in COLUMN_ORDER:
    if col == "ipsnr":
        continue
    vals = [r.get(col) for _, _, r in summary_rows
             if isinstance(r.get(col), (int, float)) and not pd.isna(r.get(col))]
    summary_best_per_col[col] = max(vals) if vals else None

summary_lines = []
summary_lines.append(r"\begin{table}[htbp]")
summary_lines.append(r"\centering")
summary_lines.append(r"\footnotesize")
summary_lines.append(r"\caption{Best-performing configuration (by AUPRC) within each architecture/pipeline group, for direct cross-group comparison. Stage 1 and Stage 2 of the cascaded HAUNet pipeline predict different targets (VIIRS and Landsat respectively) and are therefore listed as separate rows rather than compared directly.}")
summary_lines.append(r"\label{tab:best_overall}")
summary_lines.append(r"\begin{tabularx}{\textwidth}{lYYYYYY}")
summary_lines.append(r"\hline")
summary_header = " & ".join(["\\textbf{Group}"] + [f"\\textbf{{{COLUMN_LABELS[c]}}}" for c in COLUMN_ORDER])
summary_lines.append(summary_header + r" \\ \hline")
for header, label, row in summary_rows:
    cells = [f"{header} ({label})"]
    for col in COLUMN_ORDER:
        v = row.get(col)
        is_best = (col != "ipsnr") and (summary_best_per_col.get(col) is not None) and (v == summary_best_per_col[col])
        cells.append(fmt(v, is_best=is_best))
    summary_lines.append(" & ".join(cells) + r" \\")
summary_lines.append(r"\hline")
summary_lines.append(r"\end{tabularx}")
summary_lines.append(r"\end{table}")

full_summary_table = "\n".join(summary_lines)

In [212]:
# ------------------------------------------------------------------
# Notebook preview: flat DataFrame view with per-group best highlighted
# ------------------------------------------------------------------
preview_rows = []

for group in GROUPS:
    for eid in group["order"]:
        if eid not in experiments:
            continue
        row = experiments[eid]
        label = group["labels"].get(eid, eid)
        preview_rows.append({
            "Group": group["header"],
            "Experiment": label,
            **row,
        })

for name, b in [("Baseline: All-Zero", baseline_all_zero), ("Baseline: Prevalence-Rate", baseline_prevalence)]:
    preview_rows.append({"Group": "Baselines", "Experiment": name, **b})

preview_df = pd.DataFrame(preview_rows).set_index(["Group", "Experiment"])
preview_df = preview_df.reindex(columns=COLUMN_ORDER)
preview_df.columns = [COLUMN_LABELS[c] for c in COLUMN_ORDER]

numeric_preview_cols = [COLUMN_LABELS[c] for c in COLUMN_ORDER if c != "ipsnr"]

subgroup_map = {}
for g in GROUPS:
    for idx, sub in enumerate(g.get("subgroups", [g["order"]])):
        for eid in sub:
            subgroup_map[g["labels"].get(eid, eid)] = f"{g['header']}_{idx}"

def highlight_best_per_group(df):
    styles = pd.DataFrame("", index=df.index, columns=df.columns)
    mapped_groups = pd.Series(df.index.get_level_values("Experiment")).map(subgroup_map).values

    for sg_id in set(mapped_groups):
        if pd.isna(sg_id):
            continue
        mask = mapped_groups == sg_id
        group_slice = df.iloc[mask]

        for col in numeric_preview_cols:
            col_vals = group_slice[col].dropna()
            if len(col_vals) <= 1:
                continue  # no comparison possible with 0 or 1 valid values
            best_val = col_vals.max()
            for idx in col_vals[col_vals == best_val].index:
                styles.loc[idx, col] = "background-color: lightgreen"
    return styles

print("--- Consolidated Results Preview (matches exported LaTeX table) ---")
display(
    preview_df.style
    .apply(highlight_best_per_group, axis=None)
    .format({col: "{:.4f}" for col in numeric_preview_cols}, na_rep="N/A")
    .format({COLUMN_LABELS["ipsnr"]: lambda x: "N/A" if pd.isna(x) else str(x)})
)

--- Consolidated Results Preview (matches exported LaTeX table) ---


In [213]:
# ------------------------------------------------------------------
# Export
# ------------------------------------------------------------------
PROJECT_ROOT = Path("..").resolve()
output_dir = PROJECT_ROOT / "deliverables" / "tables"
output_dir.mkdir(parents=True, exist_ok=True)
tex_path = output_dir / "evaluation_table.tex"

with open(tex_path, "w") as f:
    f.write(full_latex_table)

print(f"Successfully generated consolidated LaTeX table at: {tex_path}")
print(full_latex_table)

Successfully generated consolidated LaTeX table at: /home/jupyter/irp-th25/deliverables/tables/evaluation_table.tex
\begin{table}[htbp]
\centering
\footnotesize
\caption{Model performance evaluation across feature ablations vs. fixed baselines. Precision, Recall, Boundary F1, and IPSNR are computed on Otsu-thresholded predictions. IPSNR is reported as the mean over samples with nonzero post-processed IoU, with the contributing sample count in parentheses. Bold indicates the best value for that metric among the shown ablations of the same architecture and target}
\label{tab:all_results}
\begin{tabularx}{\textwidth}{lYYYYYY}
\hline
\textbf{Exp.} & \textbf{AUPRC} & \textbf{IoU} & \textbf{Precision} & \textbf{Recall} & \textbf{Boundary F1} & \textbf{IPSNR (n)} \\ \hline
\multicolumn{7}{l}{\textit{Direct (GOES->Landsat) - HAUNet}} \\
E1: Full, No Target Dilation & 0.0103 & 0.0021 & 0.0021 & 0.3403 & 0.0030 & 0.02 (59/121) \\
E1a: No Cov. Pyramid Enc. & 0.0098 & 0.0043 & 0.0043 & 0.4605 & 0.

In [214]:
summary_tex_path = output_dir / "evaluation_summary_table.tex"
with open(summary_tex_path, "w") as f:
    f.write(full_summary_table)
print(f"Successfully generated cross-group summary table at: {summary_tex_path}")
print(full_summary_table)

Successfully generated cross-group summary table at: /home/jupyter/irp-th25/deliverables/tables/evaluation_summary_table.tex
\begin{table}[htbp]
\centering
\footnotesize
\caption{Best-performing configuration (by AUPRC) within each architecture/pipeline group, for direct cross-group comparison. Stage 1 and Stage 2 of the cascaded HAUNet pipeline predict different targets (VIIRS and Landsat respectively) and are therefore listed as separate rows rather than compared directly.}
\label{tab:best_overall}
\begin{tabularx}{\textwidth}{lYYYYYY}
\hline
\textbf{Group} & \textbf{AUPRC} & \textbf{IoU} & \textbf{Precision} & \textbf{Recall} & \textbf{Boundary F1} & \textbf{IPSNR (n)} \\ \hline
Direct - HAUNet (E1d: Dilated Target) & 0.0163 & 0.0063 & 0.0072 & 0.3101 & 0.0043 & 0.07 (55/121) \\
Cascaded Stage 1 - HAUNet (E2.1a: Stage 1 (no dilation)) & \textbf{0.1458} & \textbf{0.0651} & \textbf{0.0721} & 0.3582 & \textbf{0.0956} & 0.62 (807/1564) \\
Cascaded Stage 2 - HAUNet (E2.2a: Stage 2 (E2.1a

### Check ipsnr

In [60]:
import pandas as pd
from pathlib import Path

LOGS_DIR = Path("../logs")

configs = {
    "Full": "e1_haunet_direct_focal_dice_best_val_auprc_test_per_sample.csv",
    "Dilated": "e1_haunet_direct_focal_dice_dilated_best_val_auprc_test_per_sample.csv",
    "No Cov Pyramid": "e1_haunet_direct_focal_dice_no_cov_encoder_best_val_auprc_test_per_sample.csv",
    "No DEM": "e1_haunet_direct_focal_dice_no_dem_best_val_auprc_test_per_sample.csv",
    "No Land Cover": "e1_haunet_direct_focal_dice_no_landcover_best_val_auprc_test_per_sample.csv",
}

for name, fname in configs.items():
    path = LOGS_DIR / fname
    if not path.exists():
        print(f"[skip] {name}: {fname} not found")
        continue
    df = pd.read_csv(path)

    n_total = len(df)
    n_nonzero_ipsnr = df["ipsnr"].notna().sum()   # non-NaN = had a valid intersection
    n_nonzero_iou = (df["iou"] > 0).sum()          # samples with any true positive overlap at all

    print(f"\n--- {name} ---")
    print(f"  n_samples: {n_total}")
    print(f"  n_samples with valid (non-NaN) ipsnr: {n_nonzero_ipsnr}")
    print(f"  n_samples with iou > 0: {n_nonzero_iou}")
    print(f"  ipsnr mean (all valid): {df['ipsnr'].mean():.4f}")
    print(f"  ipsnr mean (only iou>0 samples): {df.loc[df['iou'] > 0, 'ipsnr'].mean():.4f}")
    print(f"  ipsnr median: {df['ipsnr'].median():.4f}")
    print(f"  ipsnr max: {df['ipsnr'].max():.4f}  (sample_id: "
          f"{df.loc[df['ipsnr'].idxmax(), 'sample_id'] if n_nonzero_ipsnr > 0 else 'N/A'})")
    print(f"  top 5 ipsnr values: {df['ipsnr'].nlargest(5).tolist()}")


--- Full ---
  n_samples: 121
  n_samples with valid (non-NaN) ipsnr: 121
  n_samples with iou > 0: 2
  ipsnr mean (all valid): 0.3618
  ipsnr mean (only iou>0 samples): 1.9527
  ipsnr median: 0.0442
  ipsnr max: 4.2790  (sample_id: LC08_L1TP_008052_20200813_20200813_01_RT_intersection_p00394)
  top 5 ipsnr values: [4.279021, 3.692351, 3.600191, 3.150324, 2.388698]

--- Dilated ---
  n_samples: 121
  n_samples with valid (non-NaN) ipsnr: 121
  n_samples with iou > 0: 5
  ipsnr mean (all valid): 0.3997
  ipsnr mean (only iou>0 samples): 2.7408
  ipsnr median: 0.0062
  ipsnr max: 4.6248  (sample_id: LC08_L1TP_033042_20200812_20200812_01_RT_intersection_p00174)
  top 5 ipsnr values: [4.624831, 4.547169, 4.480926, 4.271442, 4.258946]

--- No Cov Pyramid ---
  n_samples: 121
  n_samples with valid (non-NaN) ipsnr: 121
  n_samples with iou > 0: 4
  ipsnr mean (all valid): 0.4840
  ipsnr mean (only iou>0 samples): 3.2050
  ipsnr median: 0.0381
  ipsnr max: 5.2545  (sample_id: LC08_L1TP_22008

This means that ipsnr is very sensitive to the frequency of which the model predictions overlap with the targets.